<a href="https://colab.research.google.com/github/QuangDuy1512/ecommerce-behavior-analysis/blob/main/notebooks/04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/personal_projects/clean_dataset.csv")
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,HourKey,Category,Sub_Category,Product_Group
0,2019-12-07 18:52:21+00:00,view,20100170,2232732110089618156,apparel.trousers,nika,88.81,575358172,d860e8d1-6364-4117-a85b-cf467df2e05f,2019-12-07 18,apparel,trousers,Unknown
1,2019-12-29 10:30:43+00:00,view,12400194,2232732092087664982,electronics.audio.microphone,crown,84.94,542328720,e345748e-173f-4500-b300-64d230b07e6c,2019-12-29 10,electronics,audio,microphone
2,2019-12-18 18:43:25+00:00,view,4300183,2053013552385491165,appliances.sewing_machine,electrolux,90.07,514113327,35207372-6743-4834-8179-5488ca42b5aa,2019-12-18 18,appliances,sewing_machine,Unknown
3,2019-12-02 10:15:35+00:00,view,4000175,2053013566142809077,construction.tools.generator,polaris,77.20,576024739,6ec31ca0-bcab-4a66-ab0d-8ddd96e1ae2b,2019-12-02 10,construction,tools,generator
4,2019-12-16 23:31:13+00:00,view,31501161,2232732115617710964,apparel.shoes.keds,luminarc,107.85,588048847,0b624f68-eaa0-4a5e-a1df-5af55044e73c,2019-12-16 23,apparel,shoes,keds
...,...,...,...,...,...,...,...,...,...,...,...,...,...
674344,2019-12-15 15:28:48+00:00,view,12100567,2053013555397001616,appliances.kitchen.mixer,Unknown,45.24,575519990,44e1c585-1615-4782-9786-13b5e39b418e,2019-12-15 15,appliances,kitchen,mixer
674345,2019-12-19 09:28:21+00:00,view,100019496,2232732061804790604,furniture.bedroom.bed,ikea,210.30,588828669,eb54dd0a-35e1-42de-b581-b10d482a58ac,2019-12-19 09,furniture,bedroom,bed
674346,2019-12-08 14:17:52+00:00,view,1004961,2232732093077520756,construction.tools.light,oppo,154.42,516349554,06ac90ed-e6e8-4851-838b-663631eddd94,2019-12-08 14,construction,tools,light
674347,2019-12-31 15:01:29+00:00,view,100014343,2053013563835941749,appliances.kitchen.refrigerators,oneplus,1093.98,512455128,6ab74a83-5563-4e1a-91c6-da725ae18896,2019-12-31 15,appliances,kitchen,refrigerators


In [ ]:
# Đặc trưng thời gian
df["event_time"] = pd.to_datetime(df["event_time"]).dt.tz_localize(None)
df["Day"] = df["event_time"].dt.day
df["Hour"] = df["event_time"].dt.hour
df["DayOfWeek_Num"] = df["event_time"].dt.weekday
df["Weekday"] = (df["event_time"].dt.weekday)

# Tạo biến cờ xác định ngày cuối tuần (5: Thứ Bảy, 6: Chủ Nhật)
df["Is_Weekend"] = (df["Weekday"] >= 5).astype(int)

# Biến đổi chu kỳ giờ (Cyclic Encoding) bằng Sin/Cos
df["Hour_Sin"] = np.sin(24 * np.pi * df["Hour"] / 24)
df["Hour_Cos"] = np.cos(24 * np.pi * df["Hour"] / 24)

In [ ]:
# Tổng số hành động trong 1 phiên (Độ dài hành trình)
df["Session_Action_Count"] = df.groupby("user_session")["user_session"].transform("size")

# Mức độ hot/phổ biến của sản phẩm trên sàn
df["Product_Popularity"] = df.groupby("product_id")["product_id"].transform("size")

# Mức độ "nghiện" hoặc tích cực của User trên sàn
df["User_Activity"] = df.groupby("user_id")["user_id"].transform("size")

In [ ]:
# Biến mục tiêu (Target Variable) cho bài toán Phân loại ML: Khách hàng có Mua hay Không?
df["Purchase_Flag"] = (df["event_type"] == "purchase").astype(int)

# Mã hóa hành vi phục vụ làm tính năng đầu vào cho ML
event_map = {"view": 1, "cart": 2, "purchase": 3}
df["Event_Score"] = df["event_type"].map(event_map)

In [ ]:
# Giảm độ lệch phải (Right-skewed) nặng của cột giá bằng Log Transformation
df["Log_Price"] = np.log1p(df["price"])

In [ ]:
#  SIMULATED A/B TESTING FRAMEWORK
np.random.seed(42)
unique_users = df["user_id"].unique()

# Map nhóm cố định theo User_ID để giữ tính nhất quán của thực nghiệm
group_map = {user: np.random.choice(["Control", "Treatment"]) for user in unique_users}
df["AB_Group"] = df["user_id"].map(group_map)

# Mã hóa biến phân loại AB_Group thành số để ML có thể đọc trực tiếp (0 và 1)
df["AB_Group_Encoded"] = df["AB_Group"].map({"Control": 0, "Treatment": 1})

In [ ]:
# Tính toán lượng event theo HourKey để tìm điểm bất thường hệ thống
hourly_activity = df.groupby("HourKey").size().reset_index(name="Hourly_Events")

mean_events = hourly_activity["Hourly_Events"].mean()
std_events = hourly_activity["Hourly_Events"].std()

# Thiết lập ngưỡng cảnh báo bất thường vượt quá 2 lần độ lệch chuẩn
upper_threshold = mean_events + (2 * std_events)
lower_threshold = max(0, mean_events - (2 * std_events))

# Ráp ngược lại vào bảng chính để làm biến cờ cho thuật toán ML/Dashboard
df = df.merge(hourly_activity, on="HourKey", how="left")
df["Is_Anomaly_Alert"] = ((df["Hourly_Events"] > upper_threshold) | (df["Hourly_Events"] < lower_threshold)).astype(int)

In [ ]:
print("\n=== BÁO CÁO KẾT QUẢ FEATURE ENGINEERING ===")
print(f"Tổng số dòng dữ liệu hiện tại : {len(df):,}")
print(f"Tổng số cột sau khi mở zrộng  : {df.shape[1]}")
print("\nDanh sách các cột mới đã sẵn sàng cho Machine Learning:")
print(df[["Hour_Sin", "Hour_Cos", "Session_Action_Count", "Product_Popularity", "User_Activity", "Purchase_Flag", "Log_Price", "AB_Group_Encoded", "Is_Anomaly_Alert"]].head())


=== BÁO CÁO KẾT QUẢ FEATURE ENGINEERING ===
Tổng số dòng dữ liệu hiện tại : 674,349
Tổng số cột sau khi mở rộng  : 30

Danh sách các cột mới đã sẵn sàng cho Machine Learning:
       Hour_Sin  Hour_Cos  Session_Action_Count  Product_Popularity  \
0 -2.204364e-15       1.0                     1                   7   
1 -4.777360e-15       1.0                     1                   4   
2 -2.204364e-15       1.0                     1                  54   
3 -4.777360e-15       1.0                     1                   1   
4  6.369401e-15      -1.0                     1                 192   

   User_Activity  Purchase_Flag  Log_Price  AB_Group_Encoded  Is_Anomaly_Alert  
0              1              0   4.497696                 0                 0  
1              1              0   4.453649                 1                 0  
2              1              0   4.511628                 0                 0  
3              2              0   4.359270                 0             

In [ ]:
output_path = "/content/drive/MyDrive/personal_projects/enriched_ecommerce_dataset.csv"
df.to_csv(output_path, index=False)
print(f"\n[★] THÀNH CÔNG: Tập dữ liệu giàu tính năng đã được lưu tại: {output_path}")


[★] THÀNH CÔNG: Tập dữ liệu giàu tính năng đã được lưu tại: /content/drive/MyDrive/personal_projects/enriched_ecommerce_dataset.csv
